
# AI-Powered Video Editing — Motion-Aware Object Replacement (Implementation Notebook)

This notebook implements (and extends) the pipeline described in **"AI-Powered Video Editing: Motion-Aware Object Replacement Using Generative Models" (Satine Aghababyan, 2025)**.

It covers:

- **Object detection + segmentation** (YOLOv8-seg) → per-frame binary masks
- **Trajectory information propagation** (centroid + diameter endpoints) → translation/scale/rotation
- **Optical flow** (RAFT via TorchVision) → motion fields + optional occlusion masks
- **Diffusion inpainting** (Diffusers Stable Diffusion Inpainting) with:
  - **fixed latent noise** (temporal stability)
  - **ControlNet** (depth or canny structural conditioning)
  - **IP-Adapter** (previous frame as visual reference)
- **Frame warping** using optical flow (baseline temporal propagation)
- **Latent warping** (experimental; included for completeness)
- **Proposed extension**: *flow-matching latent search* — search for a latent near a base latent that minimizes optical-flow mismatch between original and inpainted consecutive frames.

> ⚠️ Compute note: Running SD + ControlNet + IP-Adapter on videos is GPU-heavy.
> The notebook is written to be runnable on a single modern GPU (e.g., 12–24GB VRAM) with careful settings.

---

## What you need

- Python 3.10+
- A CUDA GPU (recommended)
- A Hugging Face token if you download models gated by license.




## 0) Install dependencies

Run this once per environment.

If you're in Colab, you may want to restart the runtime after installation.


## Kaggle quick checklist

1) **Turn on GPU**: right sidebar → **Settings** → **Accelerator: GPU**.
2) **Turn on Internet** (needed to download Hugging Face models): right sidebar → **Settings** → toggle **Internet on**.
3) **Add data**: right sidebar → **Add data** → attach a dataset (e.g., DAVIS) or your own MP4.
4) If Stable Diffusion models are gated, add a Hugging Face token via **Add-ons → Secrets** (see the cell below).


In [ ]:
# Install deps (Kaggle-friendly)
# Kaggle comes with many preinstalled packages. Sometimes `pip install -U` upgrades shared deps
# (like pillow / pydantic / rich) and pip will warn about conflicts with those preinstalls.
# Those warnings are usually harmless for this notebook, but we pin Pillow to keep common Kaggle
# UI packages (e.g., gradio) happy.
%pip install pillow diffusers transformers accelerate safetensors ultralytics opencv-python imageio imageio-ffmpeg tqdm controlnet-aux

# Optional: if you *also* want to keep gradio/bigframes happy, you can pin these too.
# WARNING: pinning may downgrade versions and could affect other libraries.
# %pip install -q "pydantic<2.12" "rich<14"



## 1) Imports & helpers


In [ ]:

import os
import math
import json
import shutil
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
import cv2
import imageio
from tqdm.auto import tqdm

# YOLOv8 segmentation
from ultralytics import YOLO

# RAFT optical flow (TorchVision)
import torchvision
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

# Diffusers (Stable Diffusion inpainting + ControlNet)
from diffusers import (
    AutoPipelineForInpainting,
    StableDiffusionInpaintPipeline,
    StableDiffusionControlNetInpaintPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

# ControlNet auxiliary preprocessors (depth, canny, ...)
from controlnet_aux import MidasDetector, CannyDetector


def torch_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = torch_device()
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE, "dtype:", DTYPE)


def to_pil(img: np.ndarray) -> Image.Image:
    """HWC uint8 RGB -> PIL"""
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)


def pil_to_np(pil: Image.Image) -> np.ndarray:
    """PIL -> HWC uint8 RGB"""
    return np.array(pil.convert("RGB"))


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def resize_pad_to_multiple_of_8(pil: Image.Image, target_long_side: int = 768) -> Tuple[Image.Image, Tuple[int,int,int,int]]:
    """Resize keeping aspect ratio so that long side == target_long_side, then pad to multiples of 8.

    Returns: (padded_image, (pad_left, pad_top, pad_right, pad_bottom))
    """
    w, h = pil.size
    scale = target_long_side / max(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    pil_rs = pil.resize((new_w, new_h), Image.BICUBIC)

    # pad to multiple of 8
    pad_w = (8 - (new_w % 8)) % 8
    pad_h = (8 - (new_h % 8)) % 8
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top

    pil_pad = Image.new("RGB", (new_w + pad_w, new_h + pad_h), (0, 0, 0))
    pil_pad.paste(pil_rs, (pad_left, pad_top))

    return pil_pad, (pad_left, pad_top, pad_right, pad_bottom)


def unpad_and_resize(pil: Image.Image, pad: Tuple[int,int,int,int], orig_size: Tuple[int,int]) -> Image.Image:
    pad_left, pad_top, pad_right, pad_bottom = pad
    w, h = pil.size
    cropped = pil.crop((pad_left, pad_top, w - pad_right, h - pad_bottom))
    return cropped.resize(orig_size, Image.BICUBIC)


def mask_from_polygon_list(polys: List[np.ndarray], shape_hw: Tuple[int,int]) -> np.ndarray:
    """polys: list of (N,2) float/ints in image coordinates. Returns binary mask HxW uint8 {0,255}."""
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    for poly in polys:
        if poly is None or len(poly) < 3:
            continue
        pts = np.round(poly).astype(np.int32)
        cv2.fillPoly(mask, [pts], 255)
    return mask


def dilate_mask(mask: np.ndarray, k: int = 15, iters: int = 1) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.dilate(mask, kernel, iterations=iters)


def blur_mask(mask: np.ndarray, k: int = 21) -> np.ndarray:
    k = k if k % 2 == 1 else k + 1
    return cv2.GaussianBlur(mask, (k, k), 0)


def show_grid(frames: List[Image.Image], cols: int = 4, max_items: int = 12, label: Optional[str]=None) -> Image.Image:
    """Make a simple grid preview (PIL)."""
    items = frames[:max_items]
    if not items:
        raise ValueError("No frames")
    w, h = items[0].size
    rows = int(math.ceil(len(items) / cols))
    grid = Image.new("RGB", (cols * w, rows * h), (0,0,0))
    for i, im in enumerate(items):
        r, c = divmod(i, cols)
        grid.paste(im, (c * w, r * h))
    if label:
        # add a small label band on top
        band = Image.new("RGB", (grid.size[0], 40), (20,20,20))
        grid2 = Image.new("RGB", (grid.size[0], grid.size[1] + 40), (0,0,0))
        grid2.paste(band, (0,0))
        grid2.paste(grid, (0,40))
        return grid2
    return grid



## 2) Datasets (what to evaluate on)

This notebook supports **two evaluation modes**:

1. **Datasets with per-frame masks** (recommended): DAVIS, YouTube-VOS, SegTrack, KITTI-MOTS, VIPSeg, YouTube-VIS.
   - You can use *ground-truth masks* instead of YOLO.

2. **Any custom video** (fallback): we derive masks with YOLOv8-seg.

### Recommended folder convention

- Put datasets under `DATA_ROOT/`.
- Put your experiment outputs under `RUNS_ROOT/`.




### Optical-flow datasets (to validate / stress-test motion components)

If you want to evaluate **the optical-flow piece** (RAFT accuracy, flow-based warping artifacts, and the proposed *flow-matching latent search*), these are common benchmarks:

- **FlyingChairs** (synthetic; classic pretraining dataset)
- **FlyingThings3D** (synthetic; harder motion/occlusion)
- **MPI Sintel** (rendered but closer to real; includes complex motion and occlusions)
- **KITTI Optical Flow 2012/2015** (real driving scenes; sparse-ish ground truth)

> Tip: You can use these datasets purely to test your flow + warping utilities even before you run diffusion inpainting.


In [ ]:
# Kaggle paths
# - Inputs (datasets/models): /kaggle/input (read-only)
# - Writable space + outputs: /kaggle/working (up to ~20GB saved)
# DATA_ROOT='/kaggle/input/davisthesis/DAVIS-2017-trainval-Full-Resolution'
DATA_ROOT='/kaggle/input/davisthesis'
#DATA_ROOT = '/kaggle/input'
RUNS_ROOT = '/kaggle/working/runs/motion_object_replace'
ensure_dir(RUNS_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RUNS_ROOT:', RUNS_ROOT)

# List attached inputs (you should see your datasets here)
print('Attached inputs:')
print(os.listdir('/kaggle/input'))




### DAVIS loader (2016/2017)

Expected structure:

```
DAVIS/
  JPEGImages/480p/<seq_name>/<00000.jpg ...>
  Annotations/480p/<seq_name>/<00000.png ...>   # masks (palette PNG)
```

Download instructions are on the official DAVIS site (manual registration may be required).


In [ ]:

@dataclass
class VideoSequence:
    name: str
    frames: List[Image.Image]           # RGB
    masks: Optional[List[Image.Image]]  # grayscale or palette; may be None


def load_davis_sequence(davis_root: str, seq_name: str, resolution_dir: str = "Full-Resolution") -> VideoSequence:
    img_dir = os.path.join(davis_root, "JPEGImages", resolution_dir, seq_name)
    ann_dir = os.path.join(davis_root, "Annotations", resolution_dir, seq_name)

    if not os.path.isdir(img_dir):
        raise FileNotFoundError(f"Missing DAVIS frames dir: {img_dir}")

    frame_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png"))])
    frames = [Image.open(os.path.join(img_dir, f)).convert("RGB") for f in frame_files]

    masks = None
    if os.path.isdir(ann_dir):
        mask_files = sorted([f for f in os.listdir(ann_dir) if f.lower().endswith((".png",))])
        if len(mask_files) == len(frame_files):
            masks = [Image.open(os.path.join(ann_dir, f)) for f in mask_files]

    return VideoSequence(name=seq_name, frames=frames, masks=masks)



### Custom video loader

If you have a local video (mp4/mov), set `VIDEO_PATH` and we will extract frames.


In [ ]:

VIDEO_PATH = ""  # e.g. "/path/to/video.mp4" (leave empty if using a dataset)


def extract_frames_from_video(video_path: str, max_frames: Optional[int] = None, stride: int = 1) -> List[Image.Image]:
    if not os.path.isfile(video_path):
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    frames = []
    idx = 0
    kept = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
            kept += 1
            if max_frames is not None and kept >= max_frames:
                break
        idx += 1
    cap.release()
    return frames

def load_masks_from_dir(masks_dir: str, max_frames: Optional[int] = None, stride: int = 1) -> List[Image.Image]:
    """Load per-frame masks from a directory.

    Expected: files sorted lexicographically correspond to video frames.
    Supports common image formats (png/jpg). Masks will be converted to single-channel 'L'.
    """
    if not os.path.isdir(masks_dir):
        raise FileNotFoundError(masks_dir)

    exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    files = [f for f in os.listdir(masks_dir) if f.lower().endswith(exts)]
    files.sort()
    if not files:
        raise ValueError(f'No mask images found in {masks_dir}')

    masks = []
    kept = 0
    for idx, fn in enumerate(files):
        if idx % stride != 0:
            continue
        p = os.path.join(masks_dir, fn)
        m = Image.open(p).convert('L')
        masks.append(m)
        kept += 1
        if max_frames is not None and kept >= max_frames:
            break
    return masks



## 3) Object segmentation (YOLOv8-seg) OR ground-truth masks

- If your dataset provides masks, you can use them directly.
- Otherwise, we infer masks with YOLOv8 segmentation.

**Tip:** YOLO class names follow COCO for standard YOLOv8 models.


In [ ]:

# Choose your segmentation strategy
USE_GT_MASKS = True   # if your dataset has masks
USE_YOLO_MASKS = True # if no GT masks or you want to compare

# YOLO config
YOLO_MODEL = "yolov8x-seg.pt"  # smaller: yolov8n-seg.pt
TARGET_CLASS_NAME = "person"    # COCO class name, e.g. "person", "car", "dog"
YOLO_CONF = 0.25

# mask postprocess
DILATE_K = 21
DILATE_ITERS = 1
BLUR_K = 21

# SAM2 / SAM3 config
USE_SAM2_MASKS = True   # use SAM2 video tracking to generate masks
SAM2_MODEL_ID = "facebook/sam2-hiera-large"  # HF model id (or use local checkpoint+cfg)
SAM2_OBJ_ID = 1
SAM2_INIT_BOX = None  # set [x0,y0,x1,y1] in pixels on the FIRST frame (after preprocessing); None -> derive from YOLO on frame 0

# If SAM2 import fails, install once:
# %pip install -q git+https://github.com/facebookresearch/sam2.git huggingface_hub


In [ ]:

# Build COCO name->id mapping from the YOLO model metadata
# (ultralytics provides model.names)

yolo = YOLO(YOLO_MODEL)
name_to_id = {v: k for k, v in yolo.model.names.items()}
if TARGET_CLASS_NAME not in name_to_id:
    print("Available classes:", list(name_to_id.keys())[:20], "...")
    raise ValueError(f"TARGET_CLASS_NAME='{TARGET_CLASS_NAME}' not found in YOLO classes")
TARGET_CLASS_ID = name_to_id[TARGET_CLASS_NAME]
print("Target class:", TARGET_CLASS_NAME, "id:", TARGET_CLASS_ID)


In [ ]:

@torch.inference_mode()
def yolo_segment_frames(frames: List[Image.Image], target_class_id: int, conf: float = 0.25) -> List[Image.Image]:
    """Return a list of PIL (L) masks matching the target class.

    Strategy: For each frame, keep polygons for detections of the target class and OR them together.
    """
    masks_pil = []

    for pil in tqdm(frames, desc="YOLOv8-seg"):
        img = pil_to_np(pil)  # RGB
        # Ultralytics accepts numpy RGB
        res = yolo.predict(img, conf=conf, verbose=False)[0]

        polys = []
        if res.masks is not None and res.boxes is not None:
            cls = res.boxes.cls.detach().cpu().numpy().astype(int)
            # res.masks.xy is a list of polygons in xy
            for i, c in enumerate(cls):
                if int(c) != int(target_class_id):
                    continue
                poly = res.masks.xy[i]
                polys.append(np.array(poly, dtype=np.float32))

        mask = mask_from_polygon_list(polys, shape_hw=img.shape[:2])
        mask = dilate_mask(mask, k=DILATE_K, iters=DILATE_ITERS)
        mask = blur_mask(mask, k=BLUR_K)
        masks_pil.append(Image.fromarray(mask).convert("L"))

    return masks_pil


### (Optional) Better masks with SAM 2 video tracking

This replaces per-frame YOLO segmentation with **stateful** SAM2 tracking: one prompt on the first frame -> masks for the whole video.


In [ ]:
import contextlib
import os, tempfile
from pathlib import Path

def _save_frames_to_dir(frames: List[Image.Image], out_dir: str) -> str:
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for i, im in enumerate(frames):
        im.save(os.path.join(out_dir, f"{i:05d}.jpg"), quality=95)
    return out_dir

@torch.inference_mode()
def _yolo_box_from_first_frame(pil0: Image.Image, target_class_id: int, conf: float = 0.25):
    """Return xyxy box from YOLO on the first frame. Picks the highest-confidence box."""
    img0 = pil_to_np(pil0)
    res0 = yolo.predict(img0, conf=conf, verbose=False)[0]
    if res0.boxes is None or len(res0.boxes) == 0:
        return None
    cls = res0.boxes.cls.detach().cpu().numpy().astype(int)
    xyxy = res0.boxes.xyxy.detach().cpu().numpy()
    confs = res0.boxes.conf.detach().cpu().numpy()
    cand = [(xyxy[i], confs[i]) for i,c in enumerate(cls) if int(c)==int(target_class_id)]
    if not cand:
        return None
    cand.sort(key=lambda t: t[1], reverse=True)
    box = cand[0][0].tolist()  # [x0,y0,x1,y1]
    return box

@torch.inference_mode()
def sam2_segment_frames(
    frames: List[Image.Image],
    init_box_xyxy=None,
    model_id: str = "facebook/sam2-hiera-large",
    obj_id: int = 1,
):
    """Run SAM2 video predictor on a folder of JPEG frames and return PIL (L) masks per frame.

    - frames: list of PIL RGB frames (we'll save them to a temp directory)
    - init_box_xyxy: [x0,y0,x1,y1] on frame 0; required unless you switch to point prompts
    """
    from sam2.sam2_video_predictor import SAM2VideoPredictor

    tmp_dir = tempfile.mkdtemp(prefix="sam2_frames_")
    _save_frames_to_dir(frames, tmp_dir)

    predictor = SAM2VideoPredictor.from_pretrained(model_id)

    use_cuda = torch.cuda.is_available()
    amp_ctx = torch.autocast("cuda", dtype=torch.bfloat16) if use_cuda else contextlib.nullcontext()

    masks_out = []
    with amp_ctx:
        state = predictor.init_state(video_path=tmp_dir, offload_video_to_cpu=True)
        predictor.add_new_points_or_box(state, frame_idx=0, obj_id=obj_id, box=init_box_xyxy)
        for frame_idx, obj_ids, video_res_masks in predictor.propagate_in_video(state):
            # obj_ids is a list of object ids; pick the one we requested
            try:
                k = list(obj_ids).index(obj_id)
            except ValueError:
                k = 0
            m = video_res_masks[k]
            # m can be [H,W] or [1,H,W]
            if isinstance(m, torch.Tensor):
                m = m.detach().cpu()
                if m.dim() == 3:
                    m = m[0]
                m = (m > 0).numpy().astype(np.uint8) * 255
            else:
                m = np.array(m)
                if m.ndim == 3:
                    m = m[0]
                m = (m > 0).astype(np.uint8) * 255
            masks_out.append(Image.fromarray(m).convert("L"))

    return masks_out



## 4) Trajectory information propagation (centroid + diameter endpoints)

This is the thesis' basic geometric motion transfer:

- Find two farthest boundary points on a mask (approx. diameter endpoints)
- Midpoint is centroid
- Between frames: estimate translation, scale ratio, and rotation

We implement it for analysis and as a baseline compositor.


In [ ]:

from itertools import combinations


def mask_to_contour(mask_01: np.ndarray) -> Optional[np.ndarray]:
    """mask_01: HxW bool or {0,1}.
    Returns contour points (N,2) in xy or None.
    """
    mask_u8 = (mask_01.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    # choose largest contour
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2)  # xy
    return pts


def farthest_points(pts_xy: np.ndarray, max_samples: int = 2000) -> Tuple[np.ndarray, np.ndarray]:
    """Brute force farthest pair (with optional random subsampling for speed)."""
    n = pts_xy.shape[0]
    if n > max_samples:
        idx = np.random.choice(n, max_samples, replace=False)
        pts = pts_xy[idx]
    else:
        pts = pts_xy

    # compute farthest pair
    max_d = -1.0
    p1 = pts[0]
    p2 = pts[-1]
    for a, b in combinations(range(len(pts)), 2):
        d = np.sum((pts[a] - pts[b]) ** 2)
        if d > max_d:
            max_d = d
            p1, p2 = pts[a], pts[b]
    return p1.astype(np.float32), p2.astype(np.float32)


def trajectory_features(mask_pil: Image.Image) -> Optional[Dict[str, Any]]:
    """Extract centroid, diameter endpoints, diameter length, angle."""
    m = np.array(mask_pil.convert("L"))
    # binarize
    m01 = (m > 128).astype(np.uint8)
    pts = mask_to_contour(m01)
    if pts is None or len(pts) < 10:
        return None

    p1, p2 = farthest_points(pts)
    centroid = (p1 + p2) / 2.0
    diam = float(np.linalg.norm(p2 - p1))
    angle = float(math.atan2(p2[1] - p1[1], p2[0] - p1[0]))  # radians

    return {"p1": p1, "p2": p2, "centroid": centroid, "diameter": diam, "angle": angle}


def relative_transform(f1: Dict[str, Any], f2: Dict[str, Any]) -> Dict[str, float]:
    """Compute translation, scale ratio, rotation (radians) from frame1 -> frame2."""
    dx, dy = (f2["centroid"] - f1["centroid"]).tolist()
    # size ratio based on squared diameter (as in thesis)
    r = (f2["diameter"] ** 2) / (f1["diameter"] ** 2 + 1e-8)
    dtheta = float(f2["angle"] - f1["angle"])
    # normalize angle to [-pi, pi]
    dtheta = (dtheta + math.pi) % (2 * math.pi) - math.pi
    return {"dx": float(dx), "dy": float(dy), "scale": float(math.sqrt(max(r, 1e-8))), "dtheta": dtheta}



## 5) Optical flow with RAFT (TorchVision)

We use `torchvision.models.optical_flow.raft_large` with pretrained weights.

Outputs:
- dense flow field **(H,W,2)** in pixel units
- optional occlusion mask (forward-backward consistency)


In [ ]:

@torch.inference_mode()
def load_raft() -> Tuple[torch.nn.Module, Any]:
    weights = Raft_Large_Weights.DEFAULT
    model = raft_large(weights=weights).to(DEVICE)
    model.eval()
    transforms = weights.transforms()
    return model, transforms


def pil_to_raft_tensor(pil: Image.Image) -> torch.Tensor:
    # raft weights transforms expect list of tensors? We'll use the built-in transforms later.
    arr = np.array(pil.convert("RGB"))
    ten = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
    return ten


def flow_to_numpy(flow: torch.Tensor) -> np.ndarray:
    """flow: (1,2,H,W) -> (H,W,2) float32"""
    flow = flow[0].permute(1, 2, 0).detach().cpu().numpy().astype(np.float32)
    return flow


def compute_occlusion_mask(flow_fwd: np.ndarray, flow_bwd: np.ndarray, thresh: float = 1.0) -> np.ndarray:
    """Forward-backward consistency check.

    occluded=True where ||fwd + warp(bwd, fwd)|| > thresh

    Returns: HxW bool
    """
    h, w, _ = flow_fwd.shape
    # warp backward flow into frame1 coords using forward flow
    bwd_warp = warp_flow(flow_bwd, flow_fwd)
    fb = flow_fwd + bwd_warp
    err = np.linalg.norm(fb, axis=2)
    return err > thresh


def warp_flow(img_or_flow: np.ndarray, flow: np.ndarray) -> np.ndarray:
    """Backward-warp img_or_flow using flow.

    If flow is from target->source, and img_or_flow is source, then dest(x)=source(x+flow(x)).

    Here we use cv2.remap: dest(x)=src(map_x, map_y)
    """
    h, w = flow.shape[:2]
    grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (grid_x + flow[..., 0]).astype(np.float32)
    map_y = (grid_y + flow[..., 1]).astype(np.float32)

    if img_or_flow.ndim == 2:
        return cv2.remap(img_or_flow, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    elif img_or_flow.ndim == 3 and img_or_flow.shape[2] in (2,3,4):
        return cv2.remap(img_or_flow, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    else:
        raise ValueError(f"Unexpected shape: {img_or_flow.shape}")


@torch.inference_mode()
def raft_flow_between(pil1: Image.Image, pil2: Image.Image, raft_model, raft_transforms) -> np.ndarray:
    # Convert to tensors
    t1 = pil_to_raft_tensor(pil1)
    t2 = pil_to_raft_tensor(pil2)

    # transforms handle resizing / normalization
    t1, t2 = raft_transforms(t1, t2)
    t1 = t1.to(DEVICE)
    t2 = t2.to(DEVICE)

    flow_list = raft_model(t1[None], t2[None])
    flow = flow_list[-1]
    return flow_to_numpy(flow)


raft_model, raft_transforms = load_raft()
print("RAFT loaded")



## 6) Control images (depth + canny)

- Depth maps: MiDaS via `controlnet_aux.MidasDetector`
- Canny edges: `controlnet_aux.CannyDetector` (pure CV)


In [ ]:

# Preprocessors
'''midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
canny = CannyDetector()


def make_depth_control(pil: Image.Image, detect_resolution: int = 512, image_resolution: int = 512) -> Image.Image:
    # midas returns a PIL (RGB) depth visualization
    # controlnet_aux typically expects multiples of 64 for best results
    img = pil.resize((image_resolution, image_resolution), Image.BICUBIC)
    depth = midas(img, detect_resolution=detect_resolution, image_resolution=image_resolution)
    return depth
def _ensure_same_size(control: Image.Image, ref: Image.Image, resample=Image.BICUBIC) -> Image.Image:
     """Diffusers ControlNet expects control_image to match the input image size exactly."""
     if control.size != ref.size:
         control = control.resize(ref.size, resample)
     return control


def make_depth_control(
    pil: Image.Image,
    detect_resolution: int = 512,
    image_resolution: Optional[int] = None,
    ) -> Image.Image:
    w, h = pil.size
    pil_rgb = pil.convert("RGB")

     # Let controlnet-aux handle aspect ratio internally.
    if image_resolution is None:
        image_resolution = max(w, h)

    depth = midas(pil_rgb, detect_resolution=detect_resolution, image_resolution=image_resolution)
    if isinstance(depth, np.ndarray):
        depth = Image.fromarray(depth)
    depth = depth.convert("RGB")
    depth = _ensure_same_size(depth, pil_rgb, resample=Image.BICUBIC)
    return depth


def make_canny_control(pil: Image.Image, low: int = 100, high: int = 200) -> Image.Image:
    #img = pil.resize((512, 512), Image.BICUBIC)
    #edges = canny(img, low_threshold=low, high_threshold=high)
    img = pil.convert("RGB")
    edges = canny(img, low_threshold=low, high_threshold=high)
    if isinstance(edges, np.ndarray):
        edges = Image.fromarray(edges)
    edges = edges.convert("RGB")
    edges = _ensure_same_size(edges, img, resample=Image.NEAREST)
    return edges
    #return edges.convert("RGB")'''

# Preprocessors
midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
canny = CannyDetector()

def _ensure_same_size(control: Image.Image, ref: Image.Image, resample=Image.BICUBIC) -> Image.Image:
    """Diffusers ControlNet expects control_image to match the input image size exactly."""
    if control.size != ref.size:
        control = control.resize(ref.size, resample)
    return control

def make_depth_control(
    pil: Image.Image,
    detect_resolution: int = 512,
    image_resolution: Optional[int] = None,
) -> Image.Image:
    """Generate a depth control image that matches `pil.size` exactly.

    We run MiDaS at a possibly smaller internal resolution for speed, then resize back to the
    frame resolution so ControlNet tensors match.
    """
    w, h = pil.size
    pil_rgb = pil.convert("RGB")

    # Let controlnet-aux handle aspect ratio internally.
    if image_resolution is None:
        image_resolution = max(w, h)

    depth = midas(pil_rgb, detect_resolution=detect_resolution, image_resolution=image_resolution)

    if isinstance(depth, np.ndarray):
        depth = Image.fromarray(depth)

    depth = depth.convert("RGB")
    depth = _ensure_same_size(depth, pil_rgb, resample=Image.BICUBIC)
    return depth

def make_canny_control(pil: Image.Image, low: int = 100, high: int = 200) -> Image.Image:
    """Generate a canny control image that matches `pil.size` exactly."""
    img = pil.convert("RGB")
    edges = canny(img, low_threshold=low, high_threshold=high)
    if isinstance(edges, np.ndarray):
        edges = Image.fromarray(edges)

    edges = edges.convert("RGB")
    # Canny is edge-like; avoid blurring edges if a resize is needed
    edges = _ensure_same_size(edges, img, resample=Image.NEAREST)
    return edges


## 7) Diffusion Inpainting Pipelines

We provide two options:

1. **StableDiffusionInpaintPipeline** (no ControlNet)
2. **StableDiffusionControlNetInpaintPipeline** (ControlNet depth or canny)

And optionally load **IP-Adapter** for visual reference conditioning.

> Notes
> - Diffusers recommends using inpainting-tuned checkpoints.
> - Inpainting pipeline supports `ip_adapter_image` (visual reference) and `latents` (fixed noise).


In [ ]:

# ---- Model IDs (change if you prefer SDXL) ----
INPAINT_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"  # mirror of runwayml inpainting

# ControlNet options
CONTROLNET_TYPE = "depth"  # "depth" or "canny" or "inpaint" (see ControlNet v1.1)
CONTROLNET_MODEL_ID = {
    "depth": "lllyasviel/sd-controlnet-depth",
    "canny": "lllyasviel/sd-controlnet-canny",
    "inpaint": "lllyasviel/control_v11p_sd15_inpaint",
}[CONTROLNET_TYPE]

USE_CONTROLNET = True
USE_IP_ADAPTER = True

# IP-Adapter weights
IP_ADAPTER_REPO = "h94/IP-Adapter"
IP_ADAPTER_SUBFOLDER = "models"
IP_ADAPTER_WEIGHT = "ip-adapter_sd15.safetensors"  # general image reference
IP_ADAPTER_SCALE = 0.6

# Inpainting sampling params
NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 7.0
STRENGTH = 0.95  # how much to modify masked region
SEED = 123

# Output
RUN_NAME = "demo_run"
OUT_DIR = os.path.join(RUNS_ROOT, RUN_NAME)
ensure_dir(OUT_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:

import inspect


def load_inpaint_pipeline() -> Any:
    if USE_CONTROLNET:
        controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL_ID, torch_dtype=DTYPE)
        pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
            INPAINT_MODEL_ID,
            controlnet=controlnet,
            torch_dtype=DTYPE,
            safety_checker=None,
        )
    else:
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            INPAINT_MODEL_ID,
            torch_dtype=DTYPE,
            safety_checker=None,
        )

    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(DEVICE)

    # Memory optimizations
    if hasattr(pipe, "enable_xformers_memory_efficient_attention"):
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception as e:
            print("xFormers not enabled:", e)

    if hasattr(pipe, "enable_model_cpu_offload") and DEVICE == "cuda":
        # Uncomment if you're OOM; it can slow things down.
        # pipe.enable_model_cpu_offload()
        pass

    if USE_IP_ADAPTER:
        # Load image prompt adapter
        pipe.load_ip_adapter(IP_ADAPTER_REPO, subfolder=IP_ADAPTER_SUBFOLDER, weight_name=IP_ADAPTER_WEIGHT)
        pipe.set_ip_adapter_scale(IP_ADAPTER_SCALE)

    return pipe


def pipe_call_inpaint(pipe, prompt: str, image: Image.Image, mask_image: Image.Image,
                      control_image: Optional[Image.Image] = None,
                      ip_adapter_image: Optional[Image.Image] = None,
                      latents: Optional[torch.Tensor] = None,
                      generator: Optional[torch.Generator] = None,
                      **kwargs):
    """Call pipe robustly across diffusers versions."""
    sig = inspect.signature(pipe.__call__)
    kw = dict(
        prompt=prompt,
        image=image,
        mask_image=mask_image,
        strength=kwargs.pop("strength", STRENGTH),
        num_inference_steps=kwargs.pop("num_inference_steps", NUM_INFERENCE_STEPS),
        guidance_scale=kwargs.pop("guidance_scale", GUIDANCE_SCALE),
        generator=generator,
        latents=latents,
    )

    '''if USE_IP_ADAPTER and ip_adapter_image is not None and "ip_adapter_image" in sig.parameters:
        kw["ip_adapter_image"] = ip_adapter_image'''
    if USE_IP_ADAPTER and "ip_adapter_image" in sig.parameters:
        if ip_adapter_image is None:
            raise ValueError(
                "USE_IP_ADAPTER=True but ip_adapter_image=None. "
                "Provide a reference image (e.g., the current frame for t==0) "
                "or set USE_IP_ADAPTER=False."
            )
        kw["ip_adapter_image"] = ip_adapter_image

    if USE_CONTROLNET and control_image is not None:
        # Different versions may call this differently
        if "control_image" in sig.parameters:
            kw["control_image"] = control_image
        elif "controlnet_conditioning_image" in sig.parameters:
            kw["controlnet_conditioning_image"] = control_image
        else:
            # fall back to positional
            return pipe(prompt, image, mask_image, control_image, **kw)

    kw.update(kwargs)
    return pipe(**kw)


pipe = load_inpaint_pipeline()
print("Loaded pipeline:", type(pipe).__name__)



## 8) Fixed latent noise (temporal consistency)

We create a **single base latent tensor** and reuse it for each frame.

- This reduces frame-to-frame stochastic variation.
- You can optionally **perturb** it per frame (for the flow-matching search).


In [ ]:

# Create a fixed generator
gen = torch.Generator(device=DEVICE)
gen.manual_seed(SEED)


def make_base_latents(pipe, height: int, width: int, generator: torch.Generator) -> torch.Tensor:
    # Diffusers uses VAE latent channels (typically 4)
    latent_channels = getattr(pipe.vae.config, "latent_channels", 4)
    latents_shape = (1, latent_channels, height // 8, width // 8)
    latents = torch.randn(latents_shape, generator=generator, device=DEVICE, dtype=DTYPE)
    return latents



## 9) Video run config

Pick **either** a DAVIS sequence **or** a custom video.

Also set the replacement prompt.


In [ ]:

# --- Choose input ---
USE_DAVIS = True
DAVIS_ROOT = os.path.join(DATA_ROOT, "DAVIS")
DAVIS_SEQ = "bear"  # e.g. "bear" or "bike-packing"

# Optional: if you use a custom VIDEO_PATH and you already have per-frame masks,\n
# upload them as a folder and set MASKS_DIR to that folder.\n
MASKS_DIR = ''  # e.g. '/kaggle/input/my-video-dataset/masks'\n

MAX_FRAMES = 30
FRAME_STRIDE = 1

# --- Prompts ---
PROMPT = "a cute robot"  # replacement object prompt
NEG_PROMPT = "blurry, low quality, worst quality, jpeg artifacts, deformed, ugly"

# Processing resolution (long side), keep aspect, then pad to /8
TARGET_LONG_SIDE = 768


In [ ]:

# Load sequence
if USE_DAVIS:
    seq = load_davis_sequence(DAVIS_ROOT, DAVIS_SEQ)
    frames = seq.frames[:MAX_FRAMES]
    gt_masks = seq.masks[:MAX_FRAMES] if seq.masks is not None else None
else:
    if not VIDEO_PATH:
        raise ValueError("Set VIDEO_PATH or enable USE_DAVIS")
    frames = extract_frames_from_video(VIDEO_PATH, max_frames=MAX_FRAMES, stride=FRAME_STRIDE)
    gt_masks = None
    if MASKS_DIR:
        gt_masks = load_masks_from_dir(MASKS_DIR, max_frames=MAX_FRAMES, stride=FRAME_STRIDE)
        print('Loaded custom masks from:', MASKS_DIR, 'count=', len(gt_masks))

print("Frames:", len(frames), "GT masks:", gt_masks is not None)
print("Original size:", frames[0].size)

# Preview a few
preview = show_grid(frames[:12], cols=4)
preview



## 10) Prepare frames/masks at model resolution

We resize & pad each frame to preserve aspect ratio and align sizes to multiples of 8.
We do the same for masks.


In [ ]:

# Resize/pad frames
frames_proc = []
pads = []
orig_sizes = []

for pil in frames:
    orig_sizes.append(pil.size)
    pil_pad, pad = resize_pad_to_multiple_of_8(pil, target_long_side=TARGET_LONG_SIDE)
    frames_proc.append(pil_pad)
    pads.append(pad)

print("Processed size:", frames_proc[0].size, "pad:", pads[0])
show_grid(frames_proc[:12], cols=4)


In [ ]:

# Prepare masks: GT if available else YOLO
masks_proc = None

if USE_GT_MASKS and gt_masks is not None:
    masks_proc = []
    for m_pil, pad, orig in zip(gt_masks, pads, orig_sizes):
        # Convert GT to binary mask (any non-zero label)
        m_np = np.array(m_pil)
        m_bin = (m_np > 0).astype(np.uint8) * 255
        m_pil_bin = Image.fromarray(m_bin).convert("L")
        m_pad, _ = resize_pad_to_multiple_of_8(m_pil_bin.convert("RGB"), target_long_side=TARGET_LONG_SIDE)
        masks_proc.append(m_pad.convert("L"))

if (masks_proc is None) and USE_SAM2_MASKS:
    # Initialize SAM2 with a box on frame 0. If SAM2_INIT_BOX is None, we derive it from YOLO.
    init_box = SAM2_INIT_BOX
    if init_box is None:
        init_box = _yolo_box_from_first_frame(frames_proc[0], TARGET_CLASS_ID, conf=YOLO_CONF)
    if init_box is None:
        raise RuntimeError("SAM2 enabled but could not derive an init box from YOLO. Set SAM2_INIT_BOX manually.")
    masks_proc = sam2_segment_frames(frames_proc, init_box_xyxy=init_box, model_id=SAM2_MODEL_ID, obj_id=SAM2_OBJ_ID)

if (masks_proc is None) and USE_YOLO_MASKS:
    masks_proc = yolo_segment_frames(frames_proc, TARGET_CLASS_ID, conf=YOLO_CONF)

if masks_proc is None:
    raise RuntimeError("No masks available (enable GT masks, SAM2 masks, or YOLO masks)")

# Preview masks
mask_rgb = [Image.merge("RGB", (m, m, m)) for m in masks_proc[:12]]
show_grid(mask_rgb, cols=4)



## 11) Compute optical flow for the original video

We compute flows between consecutive **processed** frames.

We compute both directions:
- forward: t -> t+1
- backward: t+1 -> t

Backward flow is used for backward-warping.


In [ ]:

flows_fwd = []  # len = N-1, each (H,W,2)
flows_bwd = []
occlusions = []

for t in tqdm(range(len(frames_proc) - 1), desc="RAFT flows"):
    fwd = raft_flow_between(frames_proc[t], frames_proc[t+1], raft_model, raft_transforms)
    bwd = raft_flow_between(frames_proc[t+1], frames_proc[t], raft_model, raft_transforms)
    flows_fwd.append(fwd)
    flows_bwd.append(bwd)
    occ = compute_occlusion_mask(fwd, bwd, thresh=1.5)
    occlusions.append(occ)

print("Flows computed:", len(flows_fwd))



## 12) Baselines: (A) Trajectory propagation compositor

This baseline:

1. Inpaints the first frame with the replacement object (diffusion)
2. Extracts object patch from the inpainted frame using the mask
3. Applies translation/scale/rotation transforms to the patch across time

It does **not** solve background reveal artifacts (as discussed in the thesis).

We include it primarily as a reference.


In [ ]:

def extract_patch(im_rgb: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Tuple[int,int,int,int]]:
    """Return patch RGB, patch mask, and bbox (x0,y0,x1,y1)"""
    ys, xs = np.where(mask > 128)
    if len(xs) == 0 or len(ys) == 0:
        raise ValueError("Empty mask")
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    # pad bbox a bit
    pad = 10
    x0 = max(0, x0 - pad)
    y0 = max(0, y0 - pad)
    x1 = min(im_rgb.shape[1] - 1, x1 + pad)
    y1 = min(im_rgb.shape[0] - 1, y1 + pad)

    patch = im_rgb[y0:y1+1, x0:x1+1].copy()
    pmask = mask[y0:y1+1, x0:x1+1].copy()
    return patch, pmask, (x0, y0, x1, y1)


def apply_affine(patch: np.ndarray, pmask: np.ndarray, scale: float, dtheta: float) -> Tuple[np.ndarray, np.ndarray]:
    h, w = patch.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, math.degrees(dtheta), scale)
    patch_w = cv2.warpAffine(patch, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    mask_w  = cv2.warpAffine(pmask, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return patch_w, mask_w


def composite_patch(bg: np.ndarray, patch: np.ndarray, pmask: np.ndarray, x: int, y: int) -> np.ndarray:
    out = bg.copy()
    h, w = patch.shape[:2]
    x0 = int(round(x))
    y0 = int(round(y))

    # clip
    x1 = min(out.shape[1], x0 + w)
    y1 = min(out.shape[0], y0 + h)
    sx0 = 0
    sy0 = 0
    if x0 < 0:
        sx0 = -x0
        x0 = 0
    if y0 < 0:
        sy0 = -y0
        y0 = 0

    patch_c = patch[sy0:sy0+(y1-y0), sx0:sx0+(x1-x0)]
    mask_c  = pmask[sy0:sy0+(y1-y0), sx0:sx0+(x1-x0)]

    alpha = (mask_c.astype(np.float32) / 255.0)[..., None]
    out[y0:y1, x0:x1] = (alpha * patch_c + (1 - alpha) * out[y0:y1, x0:x1]).astype(np.uint8)
    return out


In [ ]:

# Run baseline A on a short prefix
BASELINE_A_FRAMES = 10

# Inpaint first frame with diffusion
im0 = frames_proc[0]
mask0 = masks_proc[0]

# control image for first frame
control0 = None
if USE_CONTROLNET:
    control0 = make_depth_control(im0) if CONTROLNET_TYPE == "depth" else make_canny_control(im0)

# Safety: make sure control image matches frame size (ControlNet requires this)
if USE_CONTROLNET and control0 is not None and control0.size != im0.size:
    control0 = control0.resize(im0.size, Image.BICUBIC)

print("Sizes -> image:", im0.size, "mask:", mask0.size, "control:", (control0.size if control0 is not None else None))

out0 = pipe_call_inpaint(
    pipe,
    prompt=PROMPT,
    image=im0,
    mask_image=mask0,
    control_image=control0,
    ip_adapter_image=(im0 if USE_IP_ADAPTER else None),
    latents=None,
    generator=gen,
    negative_prompt=NEG_PROMPT,
).images[0]

# extract patch from out0
out0_np = pil_to_np(out0)
mask0_np = np.array(mask0)
patch, pmask, bbox = extract_patch(out0_np, mask0_np)

# trajectory features
feat_prev = trajectory_features(mask0)
if feat_prev is None:
    raise RuntimeError("Failed to extract features from mask0")

baselineA = [out0]
centroid_prev = feat_prev["centroid"]

for t in range(1, min(BASELINE_A_FRAMES, len(frames_proc))):
    feat = trajectory_features(masks_proc[t])
    if feat is None:
        baselineA.append(frames_proc[t])
        continue

    tr = relative_transform(feat_prev, feat)

    # apply affine to patch
    patch_w, pmask_w = apply_affine(patch, pmask, scale=tr["scale"], dtheta=tr["dtheta"])

    # translate using centroid shift (approx)
    dx, dy = tr["dx"], tr["dy"]

    # bbox top-left position in frame 0
    x0, y0, x1, y1 = bbox
    new_x = x0 + dx
    new_y = y0 + dy

    bg = pil_to_np(frames_proc[t])
    comp = composite_patch(bg, patch_w, pmask_w, new_x, new_y)
    baselineA.append(to_pil(comp))

    feat_prev = feat

show_grid(baselineA, cols=4)


## 13) Baseline: (B) Optical flow warping

This reproduces the thesis' idea:

- Compute optical flow between consecutive **original** frames
- Warp the **inpainted** frame forward to approximate the next frame

We implement backward warping using the reverse flow for numerical stability.


In [ ]:

def warp_image_with_bwd_flow(src_rgb: np.ndarray, flow_tgt_to_src: np.ndarray) -> np.ndarray:
    return warp_flow(src_rgb, flow_tgt_to_src)


# Warp out0 to frame1 using backward flow (t1->t0)
if len(frames_proc) >= 2:
    out0_np = pil_to_np(out0)
    flow_1_to_0 = flows_bwd[0]  # computed between frame1->frame0
    warped1 = warp_image_with_bwd_flow(out0_np, flow_1_to_0)
    Image.fromarray(warped1)



## 14) Main method: Controlled diffusion inpainting across frames

We implement:

- **Fixed latent** (base latent)
- **ControlNet** (depth or canny)
- **IP-Adapter** reference image (previous inpainted frame)

This is the strongest *practical* method in the thesis.

### Optional: flow-guided latent search (proposed extension)

If enabled, for each frame we search around the base latent for a candidate latent that makes the inpainted optical flow closer to the original optical flow.

This implements the *minimize flow mismatch* idea without requiring differentiating through the entire diffusion process.


In [ ]:

# Proposed extension toggles
ENABLE_FLOW_MATCH_LATENT_SEARCH = True
FLOW_MATCH_ITERS = 2
FLOW_MATCH_CANDIDATES = 6
FLOW_MATCH_SIGMA = 0.15
FLOW_MATCH_MASK_ONLY = True


In [ ]:

@torch.inference_mode()
def compute_flow_loss(
    prev_inpaint: Image.Image,
    cand_inpaint: Image.Image,
    target_flow: np.ndarray,
    raft_model,
    raft_transforms,
    mask: Optional[np.ndarray] = None,
) -> float:
    """Compute MSE between target_flow and flow(prev_inpaint->cand_inpaint).

    If mask is provided (HxW bool or 0/255), compute loss only in masked region.
    """
    flow_pred = raft_flow_between(prev_inpaint, cand_inpaint, raft_model, raft_transforms)
    diff = (flow_pred - target_flow)

    if mask is not None:
        if mask.dtype != np.bool_:
            mask_bool = mask > 128
        else:
            mask_bool = mask
        # avoid empty mask
        if mask_bool.sum() > 10:
            diff = diff[mask_bool]

    return float(np.mean(diff ** 2))


@torch.inference_mode()
def flow_match_latent_search(
    pipe,
    prompt: str,
    image: Image.Image,
    mask_image: Image.Image,
    control_image: Optional[Image.Image],
    prev_inpaint: Image.Image,
    target_flow: np.ndarray,
    base_latents: torch.Tensor,
    generator: torch.Generator,
    neg_prompt: str,
    iters: int = 2,
    candidates: int = 6,
    sigma: float = 0.15,
    mask_only: bool = True,
) -> Tuple[Image.Image, torch.Tensor, float]:
    """Zero-order search for a latent near base_latents that minimizes optical-flow mismatch.

    Returns: (best_image, best_latents, best_loss)
    """

    best_lat = base_latents.clone()
    # start from base
    out = pipe_call_inpaint(
        pipe,
        prompt=prompt,
        image=image,
        mask_image=mask_image,
        control_image=control_image,
        ip_adapter_image=prev_inpaint,
        latents=best_lat,
        generator=generator,
        negative_prompt=neg_prompt,
    ).images[0]

    mask_np = np.array(mask_image) if mask_only else None
    best_loss = compute_flow_loss(prev_inpaint, out, target_flow, raft_model, raft_transforms, mask=mask_np)

    for _ in range(iters):
        for _k in range(candidates):
            noise = torch.randn_like(best_lat)#, generator=generator)
            cand_lat = best_lat + sigma * noise
            cand = pipe_call_inpaint(
                pipe,
                prompt=prompt,
                image=image,
                mask_image=mask_image,
                control_image=control_image,
                ip_adapter_image=prev_inpaint,
                latents=cand_lat,
                generator=generator,
                negative_prompt=neg_prompt,
            ).images[0]
            loss = compute_flow_loss(prev_inpaint, cand, target_flow, raft_model, raft_transforms, mask=mask_np)
            if loss < best_loss:
                best_loss = loss
                best_lat = cand_lat
                out = cand

    return out, best_lat, best_loss


In [6]:

# Run the main pipeline
N = len(frames_proc)

results = []
latents_used = []
losses = []

# Precompute model resolution for latents
H, W = frames_proc[0].size[1], frames_proc[0].size[0]  # PIL size is (W,H)
base_latents = make_base_latents(pipe, height=H, width=W, generator=gen)

prev_out = None

for t in tqdm(range(N), desc="Inpainting"):
    frame = frames_proc[t]
    mask = masks_proc[t]

    if USE_CONTROLNET:
        control = make_depth_control(frame) if CONTROLNET_TYPE == "depth" else make_canny_control(frame)
    else:
        control = None

    if t == 0:
        out = pipe_call_inpaint(
            pipe,
            prompt=PROMPT,
            image=frame,
            mask_image=mask,
            control_image=control,
            ip_adapter_image=(frame if USE_IP_ADAPTER else None),
            latents=base_latents,
            #generator=gen,
            negative_prompt=NEG_PROMPT,
        ).images[0]
        results.append(out)
        latents_used.append(base_latents.detach().cpu())
        losses.append(float("nan"))
        prev_out = out
        continue

    # target optical flow between original frames t-1 -> t
    target_flow = flows_fwd[t-1]

    if ENABLE_FLOW_MATCH_LATENT_SEARCH:
        out, lat_t, loss = flow_match_latent_search(
            pipe,
            prompt=PROMPT,
            image=frame,
            mask_image=mask,
            control_image=control,
            prev_inpaint=prev_out,
            target_flow=target_flow,
            base_latents=base_latents,
            generator=gen,
            neg_prompt=NEG_PROMPT,
            iters=FLOW_MATCH_ITERS,
            candidates=FLOW_MATCH_CANDIDATES,
            sigma=FLOW_MATCH_SIGMA,
            mask_only=FLOW_MATCH_MASK_ONLY,
        )
        losses.append(loss)
        latents_used.append(lat_t.detach().cpu())

    else:
        out = pipe_call_inpaint(
            pipe,
            prompt=PROMPT,
            image=frame,
            mask_image=mask,
            control_image=control,
            ip_adapter_image=prev_out,
            latents=base_latents,
            generator=gen,
            negative_prompt=NEG_PROMPT,
        ).images[0]
        losses.append(float("nan"))
        latents_used.append(base_latents.detach().cpu())

    results.append(out)
    prev_out = out

print("Done. Frames:", len(results))
show_grid(results[:12], cols=4)


NameError: name 'frames_proc' is not defined


## 15) Export video


In [ ]:

def save_video(frames_pil: List[Image.Image], out_path: str, fps: int = 10):
    ensure_dir(os.path.dirname(out_path))
    writer = imageio.get_writer(out_path, fps=fps, codec="libx264", quality=8)
    try:
        for im in frames_pil:
            writer.append_data(np.array(im.convert("RGB")))
    finally:
        writer.close()


out_video = os.path.join(OUT_DIR, "result.mp4")
save_video(results, out_video, fps=10)
print("Saved:", out_video)



## 16) Quantitative checks

We compute:

- Flow-matching loss (already collected if enabled)
- Simple temporal L1 difference between consecutive inpainted frames

These are not perfect metrics, but they are useful for debugging.


In [ ]:

# Temporal L1 between consecutive generated frames
l1s = []
for t in range(1, len(results)):
    a = np.array(results[t-1].convert("RGB")).astype(np.float32) / 255.0
    b = np.array(results[t].convert("RGB")).astype(np.float32) / 255.0
    l1s.append(float(np.mean(np.abs(a - b))))

print("Temporal L1 mean:", float(np.mean(l1s)))

if ENABLE_FLOW_MATCH_LATENT_SEARCH:
    # ignore nan for first
    vals = [x for x in losses[1:] if np.isfinite(x)]
    print("Flow-match MSE mean:", float(np.mean(vals)))



## 17) Notes / next research steps

If you want to go beyond this notebook:

1. Replace YOLO segmentation with higher-quality masks (e.g., SAM2 or ground-truth datasets).
2. Use a stronger motion model for non-rigid motion than centroid/diameter geometry.
3. Replace zero-order latent search with:
   - learned latent predictor (small MLP)
   - gradient-based latent optimization (requires custom differentiable pipeline)
4. Move from SD1.5 to SDXL inpainting for higher resolution.

